# 07 Fama-MacBeth 两步回归 多因子选股

## 论文参考
- **作者**: F. Cui
- **年份**: 2022
- **标题**: *Research on the Investment Strategy of New Energy Vehicle Industry Based on Multi-factor Stock Selection*
- **DOI**: 10.1145/3546607.3546611
- **摘要**: 本文采用Fama-MacBeth两步回归方法研究新能源汽车产业的多因子选股策略。第一步通过时间序列回归估计各股票的因子暴露(Beta)，第二步通过截面回归估计因子风险溢价。实证结果显示策略最高收益率达128.26%，夏普比率2.27，证明多因子模型在行业投资中的有效性。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### Fama-MacBeth (1973) 两步回归

#### 第一步: 时间序列回归 (估计因子暴露)
对每只股票 $i$，使用过去 $T$ 期数据回归:
$$r_{i,t} = \alpha_i + \sum_{k=1}^K \beta_{i,k} f_{k,t} + \epsilon_{i,t}, \quad t = 1, \ldots, T$$

得到因子暴露估计 $\hat{\beta}_{i,k}$

#### 第二步: 截面回归 (估计因子风险溢价)
在每个截面时点 $t$，用当期收益对因子暴露回归:
$$r_{i,t} = \gamma_{0,t} + \sum_{k=1}^K \gamma_{k,t} \hat{\beta}_{i,k} + \eta_{i,t}$$

$\gamma_{k,t}$ 即为因子 $k$ 在时刻 $t$ 的风险溢价估计

#### 因子风险溢价的检验
$$\bar{\gamma}_k = \frac{1}{T_2} \sum_{t=1}^{T_2} \gamma_{k,t}$$
$$t\text{-stat}(\gamma_k) = \frac{\bar{\gamma}_k}{\text{SE}(\gamma_{k,t}) / \sqrt{T_2}}$$

若 $|t\text{-stat}| > 2$，则因子风险溢价显著

### 选股策略
1. 每月末执行第一步: 用过去6个月估计每只股票的因子Beta
2. 第二步: 截面回归得到因子风险溢价
3. 使用估计的Beta和风险溢价预测下月收益
4. 选预测收益最高的Top 10等权持有

In [ ]:
# === Cell 3: Fama-MacBeth两步回归流程动画 ===
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)
n_stocks, n_periods, n_factors = 20, 60, 3
factor_names = ['动量', '波动率', 'RSI']

# Simulate factor returns
f_returns = np.random.randn(n_periods, n_factors) * 0.02
true_betas = np.random.randn(n_stocks, n_factors)
stock_returns = true_betas @ f_returns.T + np.random.randn(n_stocks, n_periods) * 0.01

# Step 1: Time-series regression -> beta estimates
from numpy.linalg import lstsq
estimated_betas = np.zeros((n_stocks, n_factors))
for i in range(n_stocks):
    X = np.column_stack([np.ones(n_periods), f_returns])
    sol = lstsq(X, stock_returns[i], rcond=None)[0]
    estimated_betas[i] = sol[1:]

# Step 2: Cross-sectional regression -> risk premiums
gammas = np.zeros((n_periods, n_factors))
for t in range(n_periods):
    X = np.column_stack([np.ones(n_stocks), estimated_betas])
    sol = lstsq(X, stock_returns[:, t], rcond=None)[0]
    gammas[t] = sol[1:]

# Animated visualization
fig = make_subplots(rows=1, cols=2,
    subplot_titles=["第一步: 因子暴露Beta估计 (时间序列)",
                    "第二步: 因子风险溢价Gamma (截面回归)"])

frames = []
for t in range(5, n_periods, 3):
    # Left: scatter of beta estimates (first 2 factors)
    frame_data = [
        go.Scatter(
            x=estimated_betas[:, 0], y=estimated_betas[:, 1],
            mode='markers+text',
            text=[f'S{i}' for i in range(n_stocks)],
            textposition='top center',
            marker=dict(size=8, color=stock_returns[:, t],
                       colorscale='RdYlGn', showscale=True,
                       colorbar=dict(title='收益率', x=0.45)),
            name='股票Beta',
        ),
        # Right: cumulative gamma time series
        go.Scatter(
            x=list(range(t)), y=np.cumsum(gammas[:t, 0]),
            mode='lines', name=factor_names[0],
            line=dict(color='#e41a1c', width=2),
        ),
        go.Scatter(
            x=list(range(t)), y=np.cumsum(gammas[:t, 1]),
            mode='lines', name=factor_names[1],
            line=dict(color='#377eb8', width=2),
        ),
        go.Scatter(
            x=list(range(t)), y=np.cumsum(gammas[:t, 2]),
            mode='lines', name=factor_names[2],
            line=dict(color='#4daf4a', width=2),
        ),
    ]
    frames.append(go.Frame(data=frame_data, name=f't={t}'))

# Initial traces
fig.add_trace(frames[0].data[0], row=1, col=1)
for trace in frames[0].data[1:]:
    fig.add_trace(trace, row=1, col=2)

fig.frames = frames
fig.update_layout(
    title=dict(text="Fama-MacBeth 两步回归过程"),
    height=500, width=1100,
    xaxis_title=f"Beta({factor_names[0]})",
    yaxis_title=f"Beta({factor_names[1]})",
    xaxis2_title="时间 (月)",
    yaxis2_title="累计风险溢价",
    updatemenus=[dict(type="buttons", buttons=[
        dict(label="\u25b6 播放", method="animate",
             args=[None, {"frame": {"duration": 400}, "transition": {"duration": 200}}]),
        dict(label="\u23f8 暂停", method="animate",
             args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}]),
    ])],
    sliders=[dict(
        steps=[dict(args=[[f.name], {"frame": {"duration": 0}, "mode": "immediate"}],
                    label=f.name, method="animate") for f in frames],
    )],
)
fig.show()

In [ ]:
# === Cell 4: 导入与配置 ===
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.api as sm

from shared.data_fetcher import get_stock_daily, get_index_daily, get_multiple_stocks_daily
from shared.factors import build_factor_panel, winsorize, standardize
from shared.backtest_engine import MultiStockBacktester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)
from shared.constants import *

print("All imports successful.")

In [ ]:
# === Cell 5: 数据获取 ===
STOCK_POOL = [
    "600519", "601318", "600036", "000858", "601166",
    "600276", "601398", "600030", "000333", "002415",
    "600900", "601888", "600809", "000568", "002304",
    "601012", "600031", "603259", "600585", "000661",
]

stock_data = get_multiple_stocks_daily(STOCK_POOL, DEFAULT_START, DEFAULT_END)
benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)

print(f"Successfully loaded {len(stock_data)} stocks")
print(f"Benchmark: {len(benchmark)} trading days")

In [ ]:
# === Cell 6: 因子工程 ===
panel = build_factor_panel(stock_data)
print(f"Factor panel shape: {panel.shape}")

FEATURE_COLS = [
    'mom_5', 'mom_10', 'mom_20', 'mom_60',
    'vol_5', 'vol_10', 'vol_20',
    'price_to_ma_5', 'price_to_ma_10', 'price_to_ma_20',
    'rsi', 'macd_hist', 'bb_pctb', 'bb_width', 'vp_corr',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in panel.columns]
TARGET_COL = 'fwd_return_20d'

for col in FEATURE_COLS:
    panel[col] = panel.groupby('date')[col].transform(
        lambda x: standardize(winsorize(x))
    )

panel.dropna(subset=FEATURE_COLS + [TARGET_COL], inplace=True)
panel['date'] = pd.to_datetime(panel['date'])
panel['year_month'] = panel['date'].dt.to_period('M')
print(f"After cleaning: {panel.shape}")
print(f"Features: {len(FEATURE_COLS)}")

In [ ]:
# === Cell 7: Fama-MacBeth 两步回归 ===
months = sorted(panel['year_month'].unique())
TRAIN_WINDOW = 6
TOP_N = 10

selections_fm = {}
gamma_history = []  # Factor risk premiums over time
tstat_history = []  # t-statistics

for i in range(TRAIN_WINDOW, len(months) - 1):
    train_months = months[i - TRAIN_WINDOW:i]
    pred_month = months[i]

    train_df = panel[panel['year_month'].isin(train_months)]
    pred_df = panel[panel['year_month'] == pred_month]

    if len(pred_df) < 5:
        continue

    symbols_in_period = list(set(train_df['symbol'].unique()) & set(pred_df['symbol'].unique()))
    if len(symbols_in_period) < 8:
        continue

    # === STEP 1: Time-series regression -> estimate betas ===
    # For each stock, regress its monthly factor values on its return
    beta_estimates = {}
    for sym in symbols_in_period:
        sym_data = train_df[train_df['symbol'] == sym]
        if len(sym_data) < 20:
            continue

        X_ts = sm.add_constant(sym_data[FEATURE_COLS].values)
        y_ts = sym_data[TARGET_COL].values

        try:
            model_ts = sm.OLS(y_ts, X_ts).fit()
            beta_estimates[sym] = model_ts.params[1:]  # exclude intercept
        except Exception:
            continue

    if len(beta_estimates) < 8:
        continue

    # === STEP 2: Cross-sectional regression -> estimate risk premiums ===
    # Get the prediction month's data
    cs_symbols = [s for s in beta_estimates if s in pred_df['symbol'].values]
    if len(cs_symbols) < 5:
        continue

    # Build cross-sectional data
    betas_array = np.array([beta_estimates[s] for s in cs_symbols])
    returns_array = []
    for s in cs_symbols:
        s_data = pred_df[pred_df['symbol'] == s]
        if len(s_data) > 0:
            returns_array.append(s_data[TARGET_COL].values[0])
        else:
            returns_array.append(0.0)
    returns_array = np.array(returns_array)

    X_cs = sm.add_constant(betas_array)
    try:
        model_cs = sm.OLS(returns_array, X_cs).fit()
        gammas = model_cs.params[1:]  # factor risk premiums
        tstats = model_cs.tvalues[1:]

        gamma_history.append(gammas)
        tstat_history.append(tstats)

        # Predict returns using betas and estimated risk premiums
        predicted_returns = {}
        for sym in cs_symbols:
            pred_ret = beta_estimates[sym] @ gammas
            predicted_returns[sym] = pred_ret

        # Select top N
        sorted_stocks = sorted(predicted_returns.items(), key=lambda x: x[1], reverse=True)
        top_stocks = [s[0] for s in sorted_stocks[:TOP_N]]

        rebal_date = pred_df['date'].max()
        selections_fm[rebal_date] = top_stocks

    except Exception as e:
        continue

print(f"Fama-MacBeth: {len(selections_fm)} rebalance periods")
print(f"Gamma samples: {len(gamma_history)}")

# Factor risk premium significance test
if gamma_history:
    gamma_arr = np.array(gamma_history)
    mean_gamma = gamma_arr.mean(axis=0)
    se_gamma = gamma_arr.std(axis=0) / np.sqrt(len(gamma_arr))
    t_stats_final = mean_gamma / np.where(se_gamma > 0, se_gamma, 1e-10)

    print("\n=== Factor Risk Premium Significance ===")
    for j, col in enumerate(FEATURE_COLS):
        sig = '***' if abs(t_stats_final[j]) > 2.58 else ('**' if abs(t_stats_final[j]) > 1.96 else '')
        print(f"  {col:20s}: gamma={mean_gamma[j]:.6f}, t-stat={t_stats_final[j]:.2f} {sig}")

In [ ]:
# === Cell 8: 回测 ===
all_prices = {sym: stock_data[sym]['close'] for sym in stock_data}
bench_close = benchmark['close'] if 'close' in benchmark.columns else None

bt = MultiStockBacktester(initial_capital=INITIAL_CAPITAL, rebalance_freq='M')
result_fm = bt.run(all_prices, selections_fm, bench_close)

print("=== Fama-MacBeth Strategy ===")
for k, v in result_fm['metrics'].items():
    print(f"  {k}: {v}")

In [ ]:
# === Cell 9: 可视化 ===
import matplotlib.pyplot as plt

# 1. 收益曲线 vs 基准
bench_eq = None
if result_fm.get('benchmark_returns') is not None:
    bench_eq = INITIAL_CAPITAL * (1 + result_fm['benchmark_returns']).cumprod()
plot_equity_curve(result_fm['equity_curve'], bench_eq,
                  title="Fama-MacBeth 策略 vs 沪深300")

# 2. 回撤
plot_drawdown(result_fm['equity_curve'], title="Fama-MacBeth 策略回撤")

# 3. 绩效表
plot_metrics_table(result_fm['metrics'], title="Fama-MacBeth 绩效指标")

# 4. 因子风险溢价时序 (累计)
if gamma_history:
    gamma_df = pd.DataFrame(gamma_history, columns=FEATURE_COLS)
    cum_gamma = gamma_df.cumsum()

    fig, ax = plt.subplots(figsize=(14, 6))
    for col in cum_gamma.columns[:8]:  # Show top 8
        ax.plot(cum_gamma[col].values, label=col, linewidth=1.5)
    ax.set_title('因子风险溢价累计值 (Gamma)', fontsize=14, fontweight='bold')
    ax.set_xlabel('调仓期数', fontsize=12)
    ax.set_ylabel('累计Gamma', fontsize=12)
    ax.legend(fontsize=9, ncol=2)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 5. 因子风险溢价t统计量
if gamma_history:
    fig, ax = plt.subplots(figsize=(12, 6))
    gamma_arr = np.array(gamma_history)
    mean_gamma = gamma_arr.mean(axis=0)
    se_gamma = gamma_arr.std(axis=0) / np.sqrt(len(gamma_arr))
    t_stats_final = mean_gamma / np.where(se_gamma > 0, se_gamma, 1e-10)

    colors = ['#4CAF50' if abs(t) > 1.96 else '#BDBDBD' for t in t_stats_final]
    bars = ax.bar(FEATURE_COLS, t_stats_final, color=colors, alpha=0.8)
    ax.axhline(y=1.96, color='red', linestyle='--', alpha=0.7, label='5%显著水平')
    ax.axhline(y=-1.96, color='red', linestyle='--', alpha=0.7)
    ax.set_title('Fama-MacBeth 因子风险溢价 t-统计量', fontsize=14, fontweight='bold')
    ax.set_ylabel('t-statistic', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 结果讨论

### Fama-MacBeth方法的核心价值
1. **因子定价检验**: 不仅选股，更重要的是检验哪些因子能获得显著风险溢价
2. **截面异质性**: 允许因子溢价随时间变化，比固定系数模型更灵活
3. **统计推断**: 提供因子溢价的t检验，有严格的统计学基础

### 两步回归的优缺点
**优点**:
- 理论基础扎实 (资产定价理论)
- 可检验因子风险溢价的统计显著性
- 处理截面相关性的经典方法

**缺点**:
- 第一步Beta估计有误差 (EIV问题)
- 要求充足的时间序列长度
- 纯线性假设，无法捕捉非线性关系

### 与论文对比
- Cui (2022) 在新能源汽车行业中获得最高128.26%收益，夏普比率2.27
- 本实验使用沪深300成分股代表性子集，行业分布更广
- 显著的因子风险溢价表明该因子在截面上具有定价能力

### 改进方向
- Shanken (1992) 校正: 修正EIV导致的t统计量偏差
- 用分组排序法替代回归作为稳健性检验
- 结合Newey-West调整处理残差自相关